# 06 · Transcription

Only filtered clips are transcribed. Backend = `transcription.backend`:

| backend | model | env |
|---|---|---|
| `qwencleo` *(default)* | QwenCleo-ASR — Egyptian Arabic + English code-switching | egyspeech-qwen |
| `cohere` | Cohere Transcribe Arabic | egyspeech |
| `parakeet` | NVIDIA NeMo FastConformer / Parakeet checkpoint | egyspeech-nemo |
| `llm` | audio LLM over an OpenAI API (vLLM + Qwen3-Omni); optional **inline tags** | egyspeech (+ vLLM server) |

For `llm`, start the server first in a terminal: `bash scripts/serve_llm.sh` (see README), and edit `transcription.llm.system_prompt` / `use_tags` in the config.
Every transcript passes sanity checks (empty, looping hallucinations, implausible speaking rate, mostly non-Arabic).

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
backend = cfg.transcription.backend
print("backend:", backend)
if backend == "llm":
    import requests
    print(requests.get(cfg.transcription.llm.url.rstrip("/") + "/models", timeout=10).json())

In [ ]:
nb.run("transcribe")

In [ ]:
nb.run("verify")   # no-op unless transcription.verify.backend is set

In [ ]:
tr = nb.transcripts_frame(lay, backend)
print(len(tr), "transcripts |", f"{tr.ok.mean():.1%} pass sanity")
display(pd.Series([f for fl in tr.flags for f in fl]).value_counts().rename("flags"))
clips = {r["id"]: r for r in nb.read_jsonl(lay.filtered)}
for _, r in tr[tr.ok].sample(min(5, int(tr.ok.sum())), random_state=1).iterrows():
    nb.play(clips[r.id]["path"], f"`{r.id}` — {r.get('text_tagged', r.text) if isinstance(r.get('text_tagged'), str) else r.text}")

In [ ]:
if "tags" in tr:
    tags = pd.Series([t for ts in tr["tags"].dropna() for t in ts]).value_counts()
    px.bar(tags, title="Tags inserted by the audio LLM").show()